In [1]:
import pandas as pd
import numpy as np
from scipy.stats import uniform, norm, beta
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import BaseEstimator
from sklearn.metrics.pairwise import haversine_distances
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import FixedThresholdClassifier
from joblib import Parallel, delayed
import xarray as xr
from yellowbrick.classifier import PrecisionRecallCurve

<h1>Data Loading</h1>

In [2]:
starts_i = pd.read_csv('final_start.csv', index_col=0)
starts_i['NDVI'] = (starts_i['B8'] - starts_i['B4']) / (starts_i['B4'] + starts_i['B8'])
starts_i.datetime_utc = pd.to_datetime(starts_i.datetime_utc)
starts_i = starts_i[['datetime_utc', 'Target', 'NDVI', 'latitude', 'longitude']]
starts_i['DOY'] = starts_i.datetime_utc.dt.dayofyear
starts = starts_i[starts_i.Target == 1]



In [3]:
ends_i = pd.read_csv('final_end.csv', index_col=0)
ends_i['NDVI'] = (ends_i['B8'] - ends_i['B4']) / (ends_i['B4'] + ends_i['B8'])
ends_i = ends_i[['datetime_utc', 'Target', 'NDVI', 'latitude', 'longitude']]
ends_i.datetime_utc = pd.to_datetime(ends_i.datetime_utc)
ends_i['DOY'] = ends_i.datetime_utc.dt.dayofyear
ends = ends_i[ends_i.Target == 1]

In [4]:
df_hourly = pd.read_csv('ERA5Land_ALL.csv', index_col=0)
df_hourly['datetime_str_utc'] = pd.to_datetime(pd.to_datetime(df_hourly.datetime_str_utc).dt.date)
df_hourly = df_hourly[df_hourly.surface_solar_radiation_downwards_hourly > 0]

In [5]:
rad_df = df_hourly.groupby(['ID', 'datetime_str_utc'])[['skin_temperature', 'temperature_2m', 'total_evaporation_hourly', 'volumetric_soil_water_layer_1']].aggregate({
  'skin_temperature':'mean', 'temperature_2m':'mean', 'total_evaporation_hourly':'mean'}).reset_index()

rad_df.index = rad_df.ID
rad_df.drop('ID', inplace=True, axis=1)
rad_df.columns = ['datetime_str_utc',	'skin_temperature', '2m_temperature', 'total_evaporation']
rad_df.datetime_str_utc = pd.to_datetime(rad_df.datetime_str_utc)
rad_df = rad_df[rad_df.datetime_str_utc.dt.year < 2025]


In [6]:
sentinel_df = pd.read_csv('sentinel_data_2019_2025.csv', index_col=0)
sentinel_df['NDVI'] = (sentinel_df['B8'] - sentinel_df['B4']) / (sentinel_df['B4'] + sentinel_df['B8'])
sentinel_df['NDMI'] = (sentinel_df['B8']  - sentinel_df['B11']) / (sentinel_df['B11'] + sentinel_df['B8'])
sentinel_df['NDWI'] = (sentinel_df['B8']  - sentinel_df['B12']) / (sentinel_df['B12'] + sentinel_df['B8'])
sentinel_df['LSWI'] = (sentinel_df['B8A']  - sentinel_df['B11']) / (sentinel_df['B11'] + sentinel_df['B8A'])
sentinel_df['NDVI3RE'] = (sentinel_df['B7'] - sentinel_df['B4']) / (sentinel_df['B4'] + sentinel_df['B7'])
sentinel_df['MSAVI2'] = (2*sentinel_df['B8A'] + 1 - np.sqrt(np.pow(2*sentinel_df['B8A']+1, 2) - 8* (sentinel_df['B8A'] - sentinel_df['B4'])))/2
sentinel_df['EVI2'] = 2.5*(sentinel_df['B8']  - sentinel_df['B4']) / (sentinel_df['B8'] + 2.4*sentinel_df['B4'] + 1)
sentinel_df = sentinel_df[['center_lat', 'center_lon', 'date'] + ['NDVI', 'NDMI', 'NDWI', 'EVI2', 'LSWI', 'NDVI3RE', 'MSAVI2']]
sentinel_df.columns = ['latitude', 'longitude', 'datetime_str_utc', 'NDVI', 'NDMI', 'NDWI', 'EVI2', 'LSWI', 'NDVI3RE', 'MSAVI2']
sentinel_df.datetime_str_utc = pd.to_datetime(sentinel_df.datetime_str_utc)

full_sent_df = pd.merge(sentinel_df, rad_df, how='inner', on=['ID', 'datetime_str_utc'])
full_sent_df['year'] = full_sent_df.datetime_str_utc.dt.year

In [7]:
df = pd.read_csv('full_data_era5_and_landsat.csv', index_col=0)
df['datetime_str_utc'] = pd.to_datetime(pd.to_datetime(df.datetime_str_utc).dt.date)
df['NDVI'] = (df['B8'] - df['B4']) / (df['B4'] + df['B8'])
df['NDMI'] = (df['B8']  - df['B11']) / (df['B11'] + df['B8'])
df['NDWI'] = (df['B8']  - df['B12']) / (df['B12'] + df['B8'])
df['LSWI'] = (df['B8A']  - df['B11']) / (df['B11'] + df['B8A'])
df['NDVI3RE'] = (df['B7'] - df['B4']) / (df['B4'] + df['B7'])
df['MSAVI2'] = (2*df['B8A'] + 1 - np.sqrt(np.pow(2*df['B8A']+1, 2) - 8* (df['B8A'] - df['B4'])))/2
df['EVI2'] = 2.5*(df['B8']  - df['B4']) / (df['B8'] + 2.4*df['B4'] + 1)
df = df[df.datetime_str_utc.dt.year < 2025]
df['year'] = df.datetime_str_utc.dt.year
df = df[['latitude', 'longitude', 'datetime_str_utc', 'NDVI', 'NDMI', 'NDWI', 'EVI2', 'LSWI', 'NDVI3RE', 'MSAVI2', 'skin_temperature', '2m_temperature', 'total_evaporation', 'year']] 


In [8]:
df = pd.concat([df, full_sent_df], axis=0)
df['year'] = df.datetime_str_utc.dt.year

In [9]:
blacklist = [10, 41, 42, 12, 14, 20, 16, 23, 27, 26, 24, 25, 38, 39, 45, 13, 21, 22, 30, 35, 33]

In [10]:
df = df[df.index.isin(set(df.index) - set(blacklist))]

In [9]:
df['ID'] = df.index.copy()
df.index = df['datetime_str_utc'].copy()

df['2m_temp_max'] = df.groupby('ID')['2m_temperature'].transform(lambda x: x.rolling('3D', min_periods=1).max())
df['2m_temp_mean'] = df.groupby('ID')['2m_temperature'].transform(lambda x: x.rolling('3D', min_periods=1).mean())
df['2m_temp_min'] = df.groupby('ID')['2m_temperature'].transform(lambda x: x.rolling('3D', min_periods=1).min())

df['skin_temp_max'] = df.groupby('ID')['skin_temperature'].transform(lambda x: x.rolling('3D', min_periods=1).max())
df['skin_temp_mean'] = df.groupby('ID')['skin_temperature'].transform(lambda x: x.rolling('3D', min_periods=1).mean())
df['skin_temp_min'] = df.groupby('ID')['skin_temperature'].transform(lambda x: x.rolling('3D', min_periods=1).min())


base_temp = 5 + 273.15

df['gdd_2m_daily'] = (df['2m_temperature'] - base_temp).clip(lower=0)
df['gdd_2m_acc'] = df.groupby(['ID', 'year'])['gdd_2m_daily'].cumsum()
df['gdd_2m_10mean'] = df.groupby(['ID', 'year'])['gdd_2m_daily'].transform(lambda x: x.rolling('10D', min_periods=1).mean())


# df['precip_acc'] = df.groupby(['ID', 'year'])['total_precipitation'].transform(lambda x: x.rolling('30D', min_periods=1).sum())

df['RedContrast'] = df['NDVI3RE'] - df['NDVI']
df['RedContrast_mean'] = df.groupby(['ID', 'year'])['RedContrast'].transform(lambda x: x.rolling('3D', min_periods=1).mean())

df['DOY'] = df.datetime_str_utc.dt.dayofyear
df['DayLength'] = 24 - 24/np.pi * np.arccos(np.tan(df['latitude']*np.pi/180) * np.tan(0.409*np.sin(2*np.pi/365*df['DOY'] - 1.39)))
df['second_half'] = np.where(df.DOY >= 184, 1, 0)

#_________________________________#
df.index = df['ID']

<h1>Feature declaration</h1>

In [10]:
start_features = ['EVI2', 'LSWI', '2m_temp_max', 'skin_temp_max', 'gdd_2m_acc', 'DayLength']
end_features = ['EVI2', 'RedContrast', '2m_temp_max', 'skin_temp_max', 'total_evaporation', 'DayLength']

<h1>Target Value creation</h1

In [11]:
l = []

start_coords_un = starts[['latitude', 'longitude']].drop_duplicates(['latitude', 'longitude']).values
end_coords_un = ends[['latitude', 'longitude']].drop_duplicates(['latitude', 'longitude']).values

for idx in df.index.unique():
  coords = np.radians(df[df.index == idx].iloc[0, [0, 1]].values.tolist())

  dist_start = np.inf
  closest_start = None
  
  for arr in start_coords_un:
    distance = haversine_distances([coords], [np.radians(arr)])
    if distance[0] < dist_start: 
      closest_start = arr.copy()
      dist_start = distance[0]
  
  dist_end = np.inf
  closest_end = None
  for arr in end_coords_un:
    distance = haversine_distances([coords], [np.radians(arr)])
    if distance[0] < dist_end: 
      closest_end = arr.copy()
      dist_end = distance[0]

  for yr in df.year.unique():
    subdf = df[(df.year == yr) & (df.index == idx)].copy()

    try:
      start = (starts[(starts.datetime_utc.dt.year == yr) & (starts.latitude == closest_start[0]) & (starts.longitude == closest_start[1])]).DOY.values[0]
    except Exception:
      start = (starts[ (starts.latitude == closest_start[0]) & (starts.longitude == closest_start[1])]).DOY.mean()
    try:
      end = (ends[(ends.datetime_utc.dt.year == yr)  & (ends.latitude == closest_end[0]) & (ends.longitude == closest_end[1])]).DOY.values[0]
    except Exception:
      end = (ends[(ends.latitude == closest_end[0]) & (ends.longitude == closest_end[1])]).DOY.mean()
    
      
    condition = (subdf.DOY >= start) & (subdf.DOY <= end)
    subdf.loc[:, 'Vegetation'] = np.where(condition, 1, 0)
    l.append(subdf)

marked_df = pd.concat(l, axis=0)



In [12]:
marked_df_start = marked_df[marked_df.second_half == 0]
marked_df_end = marked_df[marked_df.second_half == 1]

<h1>Metric and Hold-out Validation</h1>

In [13]:
def signed_distance(frame: pd.DataFrame) -> float:
  m = []
  

  df_temp = frame.copy()
  df_temp['year'] = df_temp['datetime_str_utc'].dt.year
  df_temp['DOY'] = df_temp['datetime_str_utc'].dt.dayofyear
  
  for (idx, yr), subdf in df_temp.groupby([df_temp.index, 'year']):
    
      spring = subdf[subdf['DOY'] <= 183]
      autumn = subdf[subdf['DOY'] >= 184]
      
      field_errors = []
      
      if not spring.empty:
        true_spring_active = spring[spring['Vegetation'] == 1]
        pred_spring_active = spring[spring['pred'] == 1]
        
        if len(true_spring_active) > 0 and len(pred_spring_active) > 0:
          true_start_doy = true_spring_active['DOY'].min()
          pred_start_doy = pred_spring_active['DOY'].min()
          field_errors.append(abs(true_start_doy - pred_start_doy))

      if not autumn.empty:
        true_autumn_off = autumn[autumn['Vegetation'] == 0]
        pred_autumn_off = autumn[autumn['pred'] == 0]
        
        if len(true_autumn_off) > 0 and len(pred_autumn_off) > 0:
          true_end_doy = true_autumn_off['DOY'].min()
          pred_end_doy = pred_autumn_off['DOY'].min()
          field_errors.append(abs(true_end_doy - pred_end_doy))
              
      if field_errors:
          m.append(np.mean(field_errors))
     
  return np.mean(m) if m else 25.0


In [14]:
def timeseries_CV(frame: pd.DataFrame, model: BaseEstimator, fs: list[str], hold_up_to: int = 3, max_year: int = 2025) -> None:
  
  roc = []
  f1 = []
  sd = []

  for i in range(1, hold_up_to+1):
    train = frame[frame.year <= max_year - i]
    test = frame[frame.year >= max_year - i + 1]

    X_train, X_test = train[fs], test[fs]
    y_train, y_test = train['Vegetation'], test['Vegetation']
    
    model.fit(X_train, y_train)
    
    roc.append(roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]))
    f1.append(f1_score(y_test, model.predict(X_test)))

    new = test[['datetime_str_utc', 'Vegetation']]
    new = new.assign(pred=model.predict(X_test))
  
    sd.append(signed_distance(new))

  print(f'Hold-Out Estimator: {type(model).__name__}')
  print('---------')
  print(f'Mean ROC-AUC score: {np.mean(roc)} \nMean F1-Score: {np.mean(f1)} \nMean Signed-Distance Error: {np.mean(sd)}')
  print()

In [15]:
def timeseries_CV_noprint(frame: pd.DataFrame, model: BaseEstimator, fs: list[str], hold_up_to: int = 3, max_year: int = 2025) -> float:
  
  roc = []
  f1 = []
  sd = []

  for i in range(1, hold_up_to+1):
    train = frame[frame.year <= max_year - i]
    test = frame[frame.year >= max_year - i + 1]

    X_train, X_test = train[fs], test[fs]
    y_train, y_test = train['Vegetation'], test['Vegetation']
    
    model.fit(X_train, y_train)
    
    roc.append(roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]))
    f1.append(f1_score(y_test, model.predict(X_test)))

    new = test[['datetime_str_utc', 'Vegetation']]
    new = new.assign(pred=model.predict(X_test))
  
    sd.append(signed_distance(new))
  
  return np.mean(sd)

<h1>Hold-out testing</h1>

In [16]:
# TWO MODELS - START #
forest_start = FixedThresholdClassifier(RandomForestClassifier(max_depth=6, criterion='gini', class_weight=None, min_samples_leaf=0.02, min_samples_split = 0.021, random_state=42), threshold=0.5)

timeseries_CV(marked_df_start, forest_start, start_features, 5, 2024)

Hold-Out Estimator: FixedThresholdClassifier
---------
Mean ROC-AUC score: 0.9975023656176571 
Mean F1-Score: 0.9536856008857726 
Mean Signed-Distance Error: 3.251687266876297



In [18]:
# TWO MODELS - END #

forest_end = FixedThresholdClassifier(RandomForestClassifier(max_depth=6, class_weight='balanced', criterion='gini', min_samples_leaf=0.012, min_samples_split = 0.02, random_state=42), threshold=0.5)

timeseries_CV(marked_df_end, forest_end, end_features, 5, 2024)

Hold-Out Estimator: FixedThresholdClassifier
---------
Mean ROC-AUC score: 0.9996168245809975 
Mean F1-Score: 0.9926117637776548 
Mean Signed-Distance Error: 0.5861238209225359



<h1>Spatial CV & Spatial testing</h1>

In [19]:
def spatial_CV(frame: pd.DataFrame, model: BaseEstimator, fs: list[str], test_size: int = 5) -> None:
  
  roc = []
  f1 = []
  sd = []
  
  rng = np.random.default_rng()
  indices = rng.choice(frame.index.unique(), size = len(frame.index.unique()))
  
  for i in range(1, round(len(indices)/test_size), test_size):
    test = frame[frame.index.isin(indices[i:i+test_size])]
    train = frame[frame.index.isin(indices[:i].tolist() + indices[i+test_size:].tolist())]

    X_train, X_test = train[fs], test[fs]
    y_train, y_test = train['Vegetation'], test['Vegetation']
    
    model.fit(X_train, y_train)
    
    roc.append(roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]))
    f1.append(f1_score(y_test, model.predict(X_test)))
    
    new = test[['datetime_str_utc', 'Vegetation']]
    new = new.assign(pred=model.predict(X_test))
  
    sd.append(signed_distance(new))
  
  print(f'CV Estimator: {type(model).__name__}')
  print('---------')
  print(f'Mean ROC-AUC score: {np.mean(roc)} \nMean F1-Score: {np.mean(f1)} \nMean Signed-Distance Error: {np.mean(sd)}')
  print()

In [20]:
def spatial_CV_noprint(frame: pd.DataFrame, model: BaseEstimator, fs: list[str], test_size: int = 5) -> float:
  
  roc = []
  f1 = []
  sd = []
  
  rng = np.random.default_rng()
  indices = rng.choice(frame.index.unique(), size = len(frame.index.unique()))
  
  for i in range(1, round(len(indices)/test_size), test_size):
    test = frame[frame.index.isin(indices[i:i+test_size])]
    train = frame[frame.index.isin(indices[:i].tolist() + indices[i+test_size:].tolist())]

    X_train, X_test = train[fs], test[fs]
    y_train, y_test = train['Vegetation'], test['Vegetation']
    
    model.fit(X_train, y_train)
    
    roc.append(roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]))
    f1.append(f1_score(y_test, model.predict(X_test)))
    
    new = test[['datetime_str_utc', 'Vegetation']]
    new = new.assign(pred=model.predict(X_test))
  
    sd.append(signed_distance(new))
  
  return np.mean(sd)

In [22]:
#---STARTS---#

spatial_CV(marked_df_start, forest_start, start_features, 5)

CV Estimator: FixedThresholdClassifier
---------
Mean ROC-AUC score: 0.9985158667135027 
Mean F1-Score: 0.9565391213958407 
Mean Signed-Distance Error: 2.3152173913043477



In [23]:
#---ENDS---#

spatial_CV(marked_df_end, forest_end, end_features, 5)

CV Estimator: FixedThresholdClassifier
---------
Mean ROC-AUC score: 0.9999743270477389 
Mean F1-Score: 0.9977558228501272 
Mean Signed-Distance Error: 0.12222222222222223



<h1>Grid Searching w/ averaging</h1>

In [75]:
from itertools import product

def timeseries_CV_adapted(frame: pd.DataFrame, model: BaseEstimator, fs: list[str], hold_up_to: int = 3, max_year: int = 2025, metric: str = 'f1') -> None:
  
  roc = []
  f1 = []
  sd = []
  for i in range(1, hold_up_to+1):
    train = frame[frame.year <= max_year - i]
    test = frame[frame.year >= max_year - i + 1]

    X_train, X_test = train[fs], test[fs]
    y_train, y_test = train['Vegetation'], test['Vegetation']
    
    model.fit(X_train, y_train)
    
    roc.append(roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]))
    f1.append(f1_score(y_test, model.predict(X_test)))
    if metric.lower() == 'sig_dist':
      new = test[['datetime_str_utc', 'Vegetation']]
      new = new.assign(pred=model.predict(X_test))
    
      sd.append(signed_distance(new))

  if metric.lower() == 'f1':
    return np.mean(f1)
  elif metric.lower() == 'roc':
    return np.mean(roc)
  elif metric.lower() == 'sig_dist':
    return np.mean(sd)
  
  
  
def spatial_CV_adapted(frame: pd.DataFrame, model: BaseEstimator, fs: list[str], test_size: int = 5, metric: str = 'f1') -> None:
  
  roc = []
  f1 = []
  sd=[]
  
  rng = np.random.default_rng()
  indices = rng.choice(frame.index.unique(), size = len(frame.index.unique()))
  
  for i in range(1, round(len(indices)/test_size), test_size):
    test = frame[frame.index.isin(indices[i:i+test_size])]
    train = frame[frame.index.isin(indices[:i].tolist() + indices[i+test_size:].tolist())]

    X_train, X_test = train[fs], test[fs]
    y_train, y_test = train['Vegetation'], test['Vegetation']
    
    model.fit(X_train, y_train)
    
    roc.append(roc_auc_score(y_test, model.predict(X_test)))
    f1.append(f1_score(y_test, model.predict(X_test)))
    if metric.lower() == 'sig_dist':
      new = test[['datetime_str_utc', 'Vegetation']]
      new = new.assign(pred=model.predict(X_test))

      sd.append(signed_distance(new))
    
  if metric.lower() == 'f1':
    return np.mean(f1)
  elif metric.lower() == 'roc':
    return np.mean(roc)
  elif metric.lower() == 'sig_dist':
    return np.mean(sd)
  
  
def GridSearchTrees(frame: pd.DataFrame, model_name: str, fs: list[str], params_grid: dict, metric: str = 'f1', hold_up_to: int = 3, spatial_size: int = 5, max_year: int = 2025) -> dict:
  keys, values = zip(*params_grid.items())
  param_combinations = [dict(zip(keys, v)) for v in product(*values)]
  if metric != 'sig_dist':
    best = 0
  else:
    best = np.inf
  best_params = None
  
  for params in param_combinations:
    if model_name.lower() == 'tree':
      model = DecisionTreeClassifier(random_state=42, **params)
    elif model_name.lower() == 'forest':
      model = RandomForestClassifier(random_state=42, **params)
    
    spatial = spatial_CV_adapted(frame, model, fs, spatial_size, metric)
    temporal = timeseries_CV_adapted(frame, model, fs, hold_up_to, max_year, metric)
    
    if np.mean([spatial, temporal]) > best and not metric == 'sig_dist':
      best = np.mean([spatial, temporal])
      best_params = params
    elif np.mean([spatial, temporal]) < best and metric == 'sig_dist':
      best = np.mean([spatial, temporal])
      best_params = params
      
  return best_params  
    
    

grid = {
    'min_samples_leaf': [0.01,  0.02, 0.03],
    'min_samples_split': [0.01, 0.02,  0.03],
    'criterion': ['gini', 'entropy'],
    'max_depth': [5, 6, 7],
    'class_weight': ['balanced', None]
}


In [76]:
#START#

from collections import Counter

def run_gs():
  return GridSearchTrees(marked_df_start, 'forest', start_features, grid, 'sig_dist', 5, 5, 2024)

min_samples_leaf = []
min_samples_split = []
crit=[]
depth=[]

results = Parallel(n_jobs=20)(delayed(run_gs)() for _ in range(20))

min_samples_leaf = [r['min_samples_leaf'] for r in results]
min_samples_split = [r['min_samples_split'] for r in results]
crit = [r['criterion'] for r in results]
depth = [r['max_depth'] for r in results]
weight = [r['class_weight'] for r in results]

print(f'Min_Samples_leaf: {np.mean(min_samples_leaf)}')
print(f'Min_Samples_split: {np.mean(min_samples_split)}')
print(f'Criterion: {Counter(crit).most_common(1)}')
print(f'Maximum Depth: {np.mean(depth)}')
print(f'Weights: {Counter(weight).most_common(1)}') 

Min_Samples_leaf: 0.015000000000000003
Min_Samples_split: 0.0205
Criterion: [('gini', 15)]
Maximum Depth: 5.95
Weights: [(None, 20)]


In [114]:
#END#

from collections import Counter

def run_gs():
  return GridSearchTrees(marked_df_end, 'forest', end_features, grid, 'sig_dist', 5, 5, 2024)

min_samples_leaf = []
min_samples_split = []
crit=[]
depth=[]

results = Parallel(n_jobs=20)(delayed(run_gs)() for _ in range(20))
min_samples_leaf = [r['min_samples_leaf'] for r in results]
min_samples_split = [r['min_samples_split'] for r in results]
crit = [r['criterion'] for r in results]
depth = [r['max_depth'] for r in results]
weight = [r['class_weight'] for r in results]

print(f'Min_Samples_leaf: {np.mean(min_samples_leaf)}')
print(f'Min_Samples_split: {np.mean(min_samples_split)}')
print(f'Criterion: {Counter(crit).most_common(1)}')
print(f'Maximum Depth: {np.mean(depth)}') 
print(f'Weights: {Counter(weight).most_common(1)}') 

Min_Samples_leaf: 0.0115
Min_Samples_split: 0.019000000000000003
Criterion: [('gini', 11)]
Maximum Depth: 6.1
Weights: [('balanced', 12)]


<h1>Threshold Tuning</h1>

<h3>#---Starts---#</h3>

<h5>Hold-out</h5>

In [97]:
thresholds = [0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]

best_bound = None
best_err = np.inf

model = RandomForestClassifier(max_depth=6, criterion='gini', class_weight=None, min_samples_leaf=0.015, min_samples_split = 0.0205, random_state=42)

for bound in thresholds:
  bound_model = FixedThresholdClassifier(model, threshold=bound)
  err = timeseries_CV_noprint(marked_df_start, bound_model, start_features, 5, 2024)
  if err < best_err:
    best_err = err
    best_bound = bound

print(f'Best Decision Bound: {best_bound} | Hold-Out CV')

Best Decision Bound: 0.65 | Hold-Out CV


<h5>Spatial CV</h5>

In [98]:
thresholds = [0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]

best_bound = None
best_err = np.inf
times = 20

model = RandomForestClassifier(max_depth=6, criterion='gini', class_weight=None, min_samples_leaf=0.015, min_samples_split = 0.0205, random_state=42)

for bound in thresholds:
  bound_model = FixedThresholdClassifier(model, threshold=bound)
  err = []
  for i in range(times):
    err.append(spatial_CV_noprint(marked_df_start, bound_model, start_features, 5))
  if np.mean(err) < best_err:
    best_err = np.mean(err)
    best_bound = bound

print(f'Best Decision Bound: {best_bound} | Spatial CV')

Best Decision Bound: 0.45 | Spatial CV


<h3>#---Ends---#</h3>

<h5>Hold-out</h5>

In [168]:
thresholds = [0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]

best_bound = None
best_err = np.inf

model = RandomForestClassifier(max_depth=6, class_weight='balanced', criterion='entropy', min_samples_leaf=0.01, min_samples_split = 0.025, random_state=42)
for bound in thresholds:
  bound_model = FixedThresholdClassifier(model, threshold=bound)
  err = timeseries_CV_noprint(marked_df_end, bound_model, end_features, 5, 2024)
  if err < best_err:
    best_err = err
    best_bound = bound

print(f'Best Decision Bound for the ENDS model: {best_bound} | Hold-Out CV')

Best Decision Bound for the ENDS model: 0.5 | Hold-Out CV


In [ ]:
thresholds = [0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]

best_bound = None
best_err = np.inf
times = 100
model = RandomForestClassifier(max_depth=6, class_weight='balanced', criterion='entropy', min_samples_leaf=0.01, min_samples_split = 0.025, random_state=42)
for bound in thresholds:
  bound_model = FixedThresholdClassifier(model, threshold=bound)
  err = []
  for i in range(times):
    err.append(spatial_CV_noprint(marked_df_end, bound_model, end_features, 5))
  if np.mean(err) < best_err:
    best_err = np.mean(err)
    best_bound = bound

print(f'Best Decision Bound for the ENDS model: {best_bound} | Spatial CV')

Best Decision Bound for the ENDS model: 0.45 | Spatial CV


<h1>Model Saving</h1>

In [24]:
forest_start = FixedThresholdClassifier(RandomForestClassifier(max_depth=6, criterion='gini', class_weight=None, min_samples_leaf=0.02, min_samples_split = 0.021, random_state=42), threshold=0.5)
forest_end = FixedThresholdClassifier(RandomForestClassifier(max_depth=6, class_weight='balanced', criterion='gini', min_samples_leaf=0.012, min_samples_split = 0.02, random_state=42), threshold=0.5)

marked_df_start_X = marked_df_start[start_features]
marked_df_start_y = marked_df_start['Vegetation']

marked_df_end_X = marked_df_end[end_features]
marked_df_end_y = marked_df_end['Vegetation']

forest_start.fit(marked_df_start_X, marked_df_start_y)
forest_end.fit(marked_df_end_X, marked_df_end_y)

,estimator,RandomForestC...ndom_state=42)
,threshold,0.5
,pos_label,None
,response_method,'auto'
,n_estimators,100
,criterion,'gini'
,max_depth,6
,min_samples_split,0.02
,min_samples_leaf,0.012
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'


In [25]:
import joblib

joblib.dump(forest_start, 'SOS_model.pkl')
joblib.dump(forest_end, 'EOS_model.pkl')

['EOS_model.pkl']